# NorthStar - MongoDB Atlas and Query Optimisation

This is the third and final technical notebook. It covers the MongoDB development section and the query optimisation section.

What's in here:
1. Setup and Atlas connection
2. Why MongoDB and design decisions (3 collections, embedded vs referenced)
3. Loading and preparing the data (same cleaning as the Python and R notebooks)
4. Building the three collections: `service_cases`, `asset_lifecycle`, `platform_events`
5. CRUD operations
6. Aggregation pipelines
7. Indexes
8. Query optimisation with `.explain("executionStats")` before vs after

The data is the same as the previous notebooks. The cleaning step is included here too so this notebook runs on its own.

## 1. Setup

Install pymongo and connect to MongoDB Atlas. The connection string is requested at runtime via `getpass` so credentials are never written into the notebook or committed to GitHub. When the cell runs, paste your Atlas connection string into the prompt.

To get your connection string from Atlas: Cluster -> Connect -> Drivers -> copy the string. Make sure your IP is whitelisted under Network Access and you have a database user with read/write permission.

In [ ]:
!pip install pymongo --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 410.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 324.7 kB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from pymongo import MongoClient, ASCENDING
from pymongo.errors import OperationFailure
from datetime import datetime
from pathlib import Path
import getpass

# getpass prompts the user to paste their Atlas connection string at runtime
# this avoids committing credentials to GitHub
# format: mongodb+srv://<user>:<password>@<cluster>.mongodb.net/?retryWrites=true&w=majority
CONNECTION_STRING = getpass.getpass("Paste your MongoDB Atlas connection string: ")

client = MongoClient(CONNECTION_STRING)
db = client["northstar_db"]
print("connected to:", db.name)

Paste your MongoDB Atlas connection string: ··········
connected to: northstar_db


## 2. Why MongoDB and the design

### Why a document model for parts of this problem

The case study outlines data that is already fragmented in different operating systems. The most critical findings were revealed by the SQL analysis of 4-5 tables together (complaints linked to deliveries linked to orders linked to customers, for instance). No existing NorthStar report does these joins, that's why management can't see the problems.

A document model is a collection of objects that are related to each other in a given document. The cost there is denormalisation, or multiple copies of the data in documents, but it's okay because this is the integrated view that we want.

### What goes in which collection

I designed 3 collections. The split is based on what is being queried that is split together in analysis:

**`service_cases`** - one document per customer, with their orders nested, and each order's delivery, complaints and incidents embedded inside the order. This is the one casefile that is missing from existing systems, the operational picture. A customer experience officer can have one document and get all of this information about one customer.


**`asset_lifecycle`** - one document each vehicle (incident history embedded). InRepair finding was derived from joining vehicles to deliveries to incidents - the join is trivial here.

**`platform_events`** - one document per session, with the events listed as an array. It's an obvious session oriented application (search -> book -> track -> rate) and the original flat CSV doesn't capture that.

### Embedded vs referenced

In`service_cases` and `asset_lifecycle` I used nested data instead of referring to it. Because: In the case of NorthStar, the related data is always read together. When a manager accesses a customer record, they don't want to have to ask again for the complaints, they want them to see them. The same as vehicles and incidents.

For `platform_events`, events are nested in the session document due to the natural read unit being a session. You'd never read individual events without their session context.

Instead of querying by the incident alone (no vehicle context), we would refer to them in a different collection. But the case study is on integrated case views, so embedding fits.

## 3. Load and clean the data

Same cleaning as the Python and R notebooks. Keeping it inline so this notebook can be run standalone.

In [ ]:
# data is loaded from the GitHub repo so the notebook runs end-to-end without manual upload
USE_LOCAL = False
BASE_URL  = "https://raw.githubusercontent.com/Daksh32146982/DBA-assignment/refs/heads/main/data"
LOCAL_DIR = Path('./data')

def csv_path(name):
    if USE_LOCAL and (LOCAL_DIR / f"{name}.csv").exists():
        return str(LOCAL_DIR / f"{name}.csv")
    return f"{BASE_URL}/{name}.csv"

hubs       = pd.read_csv(csv_path('hubs'))
vehicles   = pd.read_csv(csv_path('vehicles'))
drivers    = pd.read_csv(csv_path('drivers'))
customers  = pd.read_csv(csv_path('customers'))
orders     = pd.read_csv(csv_path('orders'))
deliveries = pd.read_csv(csv_path('deliveries'))
incidents  = pd.read_csv(csv_path('incidents'))
complaints = pd.read_csv(csv_path('complaints'))
app_events = pd.read_csv(csv_path('app_events'))

print("loaded raw CSVs")

loaded raw CSVs


In [ ]:
# clean zone variants
def clean_zone(v):
    if pd.isna(v): return v
    v = str(v).strip().lower()
    if v == 'ctr': v = 'central'
    return v.title()

for df, col in [(orders, 'pickup_zone'), (orders, 'dropoff_zone'),
                (drivers, 'base_zone'), (vehicles, 'assigned_zone'),
                (customers, 'home_zone'), (app_events, 'zone_context'),
                (hubs, 'zone')]:
    df[col] = df[col].apply(clean_zone)

# parse dates
for df, cols in [(orders, ['order_created_at']),
                 (deliveries, ['dispatch_time','delivery_completed_at']),
                 (incidents, ['reported_at']),
                 (complaints, ['created_at']),
                 (app_events, ['event_timestamp']),
                 (customers, ['signup_date'])]:
    for c in cols:
        df[c] = pd.to_datetime(df[c], errors='coerce')

# integrity flags - same logic as the other notebooks
deliveries['duration_hours'] = (deliveries['delivery_completed_at'] - deliveries['dispatch_time']).dt.total_seconds()/3600
deliveries['negative_duration_flag'] = (deliveries['duration_hours'] < 0).fillna(False).astype(int)
deliveries['ontime_no_completion']   = ((deliveries['delivery_status']=='OnTime') & deliveries['delivery_completed_at'].isna()).astype(int)
deliveries['failed_but_rated']       = ((deliveries['delivery_status']=='Failed') & deliveries['customer_rating_post_delivery'].notna()).astype(int)
deliveries['status_integrity_issue'] = ((deliveries['negative_duration_flag']==1) |
                                         (deliveries['ontime_no_completion']==1) |
                                         (deliveries['failed_but_rated']==1)).astype(int)

print(f"cleaning done. integrity issues flagged: {deliveries['status_integrity_issue'].sum()}")

cleaning done. integrity issues flagged: 207


## 4. Build the collections

### Helper for type conversion

MongoDB doesn't accept numpy types like `int64` or `bool_` directly - they have to be converted to native Python types. Pandas NaN/NaT also needs to become `None`. A small helper handles that.

In [ ]:
def to_py(v):
    """convert numpy/pandas types to plain python"""
    if pd.isna(v): return None
    if isinstance(v, np.integer):  return int(v)
    if isinstance(v, np.floating): return float(v)
    if isinstance(v, np.bool_):    return bool(v)
    if isinstance(v, pd.Timestamp): return v.to_pydatetime()
    return v

### Collection 1: `service_cases`

One document per customer. Embeds orders, each order embeds its delivery + complaints, each delivery embeds its incidents. This is the case file the customer experience team needs.

The document shape:

```
{
  customer_id, customer_type, home_zone, loyalty_score, ...,
  orders: [
    {
      order_id, service_type, pickup_zone, ...,
      delivery: {
        delivery_id, status, hub_id, driver_id, vehicle_id, ...,
        integrity_flags: { ... },
        incidents: [ ... ]
      },
      complaints: [ ... ]
    },
    ...
  ]
}
```

In [ ]:
db.service_cases.drop()

# precompute groupings for speed - 650 customers, doing this in a loop without
# precomputing would be much slower
orders_by_cust  = {k: g for k, g in orders.groupby('customer_id')}
deliv_by_order  = {r['order_id']: r for _, r in deliveries.iterrows()}
incs_by_deliv   = {k: g.to_dict('records') for k, g in incidents.groupby('delivery_id')}
comps_by_order  = {k: g.to_dict('records') for k, g in complaints.groupby('order_id')}

service_docs = []
for _, c in customers.iterrows():
    cust_doc = {
        'customer_id':    c['customer_id'],
        'customer_type':  c['customer_type'],
        'home_zone':      c['home_zone'],
        'signup_date':    to_py(c['signup_date']),
        'loyalty_score':  to_py(c['loyalty_score']),
        'app_engagement': to_py(c['app_engagement_score']),
        'account_status': c['account_status'],
        'orders':         []
    }

    cust_orders = orders_by_cust.get(c['customer_id'])
    if cust_orders is None:
        service_docs.append(cust_doc)
        continue

    for _, o in cust_orders.iterrows():
        order_doc = {
            'order_id':              o['order_id'],
            'service_type':          o['service_type'],
            'order_created_at':      to_py(o['order_created_at']),
            'pickup_zone':           o['pickup_zone'],
            'dropoff_zone':          o['dropoff_zone'],
            'priority_level':        o['priority_level'],
            'order_value':           to_py(o['order_value']),
            'booking_channel':       o['booking_channel'] if pd.notna(o['booking_channel']) else 'Unknown',
            'promised_window_hours': to_py(o['promised_window_hours']),
            'delivery':              None,
            'complaints':            []
        }

        # delivery nested inside the order
        if o['order_id'] in deliv_by_order:
            d = deliv_by_order[o['order_id']]
            order_doc['delivery'] = {
                'delivery_id':    d['delivery_id'],
                'driver_id':      d['driver_id'],
                'vehicle_id':     d['vehicle_id'],
                'hub_id':         d['hub_id'],
                'dispatch_time':  to_py(d['dispatch_time']),
                'completed_at':   to_py(d['delivery_completed_at']),
                'status':         d['delivery_status'],
                'duration_hours': to_py(d['duration_hours']),
                'distance_km':    to_py(d['route_distance_km']),
                'cost':           to_py(d['fuel_or_charge_cost']),
                'overrides':      to_py(d['manual_route_override_count']),
                'rating':         to_py(d['customer_rating_post_delivery']),
                'integrity_flags': {
                    'negative_duration':    bool(d['negative_duration_flag']),
                    'ontime_no_completion': bool(d['ontime_no_completion']),
                    'failed_but_rated':     bool(d['failed_but_rated']),
                    'any_issue':            bool(d['status_integrity_issue']),
                },
                'incidents': []
            }
            # incidents on this delivery
            for inc in incs_by_deliv.get(d['delivery_id'], []):
                order_doc['delivery']['incidents'].append({
                    'incident_id':       inc['incident_id'],
                    'incident_type':     inc['incident_type'],
                    'severity':          inc['severity'],
                    'resolution_status': inc['resolution_status'],
                    'reported_at':       to_py(inc['reported_at']),
                    'resolved_hours':    to_py(inc['resolved_hours']),
                })

        # complaints linked to the order
        for cm in comps_by_order.get(o['order_id'], []):
            order_doc['complaints'].append({
                'complaint_id':    cm['complaint_id'],
                'complaint_type':  cm['complaint_type'],
                'channel':         cm['channel'],
                'severity':        cm['severity'],
                'status':          cm['status'],
                'created_at':      to_py(cm['created_at']),
                'resolution_days': to_py(cm['resolution_days']),
                'compensation':    to_py(cm['compensation_amount']) if pd.notna(cm['compensation_amount']) else 0.0,
            })

        cust_doc['orders'].append(order_doc)

    service_docs.append(cust_doc)

db.service_cases.insert_many(service_docs)
print(f"service_cases: {db.service_cases.count_documents({})} documents")

service_cases: 650 documents


### Collection 2: `asset_lifecycle`

One document per vehicle with its full delivery performance summary and incident history. Lets us answer the InRepair-dispatch question in a single document read.

In [ ]:
db.asset_lifecycle.drop()

# precompute per-vehicle delivery stats
veh_stats = deliveries.groupby('vehicle_id').agg(
    deliveries=('delivery_id','count'),
    failed=('delivery_status', lambda x: (x=='Failed').sum()),
    fail_pct=('delivery_status', lambda x: round((x=='Failed').mean()*100, 1)),
    avg_overrides=('manual_route_override_count', lambda x: round(x.mean(), 2)),
    delivery_ids=('delivery_id', list),
).to_dict('index')

asset_docs = []
for _, v in vehicles.iterrows():
    stats = veh_stats.get(v['vehicle_id'], {})
    veh_doc = {
        'vehicle_id':         v['vehicle_id'],
        'vehicle_type':       v['vehicle_type'],
        'assigned_zone':      v['assigned_zone'],
        'commission_date':    to_py(v['commission_date']),
        'odometer_km':        to_py(v['odometer_km']),
        'battery_health_pct': to_py(v['battery_health_pct']),
        'telematics_version': v['telematics_version'],
        'maintenance_status': v['maintenance_status'],
        'performance': {
            'total_deliveries': int(stats.get('deliveries', 0)),
            'failed':           int(stats.get('failed', 0)),
            'fail_pct':         float(stats.get('fail_pct', 0)),
            'avg_overrides':    float(stats.get('avg_overrides', 0)),
        },
        'risk_flags': {
            'is_in_repair': v['maintenance_status'] == 'InRepair',
            'low_battery':  bool(pd.notna(v['battery_health_pct']) and v['battery_health_pct'] < 60),
            'high_mileage': bool(v['odometer_km'] > 150000),
        },
        'incidents': []
    }

    # embed incidents for this vehicle (linked via its delivery_ids)
    for did in stats.get('delivery_ids', []):
        for inc in incs_by_deliv.get(did, []):
            veh_doc['incidents'].append({
                'incident_id':       inc['incident_id'],
                'delivery_id':       did,
                'incident_type':     inc['incident_type'],
                'severity':          inc['severity'],
                'resolution_status': inc['resolution_status'],
                'reported_at':       to_py(inc['reported_at']),
                'resolved_hours':    to_py(inc['resolved_hours']),
            })
    veh_doc['risk_flags']['unresolved_incidents'] = sum(
        1 for i in veh_doc['incidents'] if i['resolution_status'] in ('Open','Escalated')
    )
    asset_docs.append(veh_doc)

db.asset_lifecycle.insert_many(asset_docs)
print(f"asset_lifecycle: {db.asset_lifecycle.count_documents({})} documents")

asset_lifecycle: 120 documents


### Collection 3: `platform_events`

The app_events.csv comes as one row per event, but events are naturally part of a session (search -> book -> pay -> track). This collection groups them by session_id so each document is a complete session.

In [ ]:
db.platform_events.drop()

session_docs = []
for sid, g in app_events.groupby('session_id'):
    g = g.sort_values('event_timestamp')
    first = g.iloc[0]
    last  = g.iloc[-1]
    duration_sec = (last['event_timestamp'] - first['event_timestamp']).total_seconds()

    sess_doc = {
        'session_id':   sid,
        'customer_id':  first['customer_id'],
        'device_type':  first['device_type'],
        'started_at':   to_py(first['event_timestamp']),
        'ended_at':     to_py(last['event_timestamp']),
        'duration_sec': float(duration_sec),
        'event_count':  int(len(g)),
        'success_rate': float(round(g['success_flag'].mean()*100, 1)),
        'avg_latency':  float(round(g['api_latency_ms'].mean(), 0)),
        'events': []
    }
    for _, e in g.iterrows():
        sess_doc['events'].append({
            'event_id':     e['event_id'],
            'event_type':   e['event_type'],
            'timestamp':    to_py(e['event_timestamp']),
            'zone_context': e['zone_context'],
            'order_id':     None if pd.isna(e['order_id']) else e['order_id'],
            'success':      bool(e['success_flag']),
            'latency_ms':   int(e['api_latency_ms']),
        })
    session_docs.append(sess_doc)

db.platform_events.insert_many(session_docs)
print(f"platform_events: {db.platform_events.count_documents({})} sessions")

platform_events: 637 sessions


In [ ]:
# quick sanity check - show one sample document from each collection
print("--- sample service_cases ---")
sample = db.service_cases.find_one({'orders.complaints.0': {'$exists': True}}, {'orders': {'$slice': 1}})
print(sample['customer_id'], '|', len(sample['orders']), 'orders shown')
if sample['orders']:
    o = sample['orders'][0]
    print(' ', o['order_id'], o['service_type'], '- delivery:', o['delivery']['status'] if o.get('delivery') else 'None')

print("\n--- sample asset_lifecycle ---")
sample = db.asset_lifecycle.find_one({'risk_flags.is_in_repair': True})
print(sample['vehicle_id'], sample['vehicle_type'], 'InRepair, deliveries=', sample['performance']['total_deliveries'])

print("\n--- sample platform_events ---")
sample = db.platform_events.find_one()
print(sample['session_id'], '|', sample['device_type'], '|', sample['event_count'], 'events')

--- sample service_cases ---
C0001 | 1 orders shown
  O00007 Business - delivery: Delayed

--- sample asset_lifecycle ---
V002 EV InRepair, deliveries= 8

--- sample platform_events ---
S10192 | Android | 1 events


## 5. CRUD operations

One example of each. Each operation is tied to a finding from the SQL/R analysis - they're not toy operations.

### CREATE - insert a test customer

In [ ]:
new_customer = {
    'customer_id':    'C_TEST_01',
    'customer_type':  'Consumer',
    'home_zone':      'North',
    'loyalty_score':  72.0,
    'app_engagement': 65.0,
    'account_status': 'Active',
    'orders':         []
}
result = db.service_cases.insert_one(new_customer)
print('inserted _id:', result.inserted_id)
print('total docs now:', db.service_cases.count_documents({}))

inserted _id: 6a0259c70e5cde680bcd49c2
total docs now: 651


### READ 1 - high-loyalty customers who have complained

These are exactly the customers most at risk of leaving. In the SQL analysis this needed a 3-table join; here it's a single document lookup.

In [ ]:
at_risk = db.service_cases.find({
    'loyalty_score': {'$gte': 70},
    'orders.complaints.0': {'$exists': True}
}).limit(10)

for c in at_risk:
    n_complaints = sum(len(o.get('complaints', [])) for o in c.get('orders', []))
    print(f"  {c['customer_id']} loyalty={c['loyalty_score']:.1f} complaints={n_complaints} status={c['account_status']}")

  C0010 loyalty=87.2 complaints=1 status=Active
  C0014 loyalty=94.1 complaints=1 status=Active
  C0023 loyalty=73.1 complaints=2 status=Active
  C0026 loyalty=70.6 complaints=2 status=Active
  C0028 loyalty=78.5 complaints=2 status=Active
  C0032 loyalty=85.0 complaints=2 status=Dormant
  C0046 loyalty=84.8 complaints=1 status=Active
  C0056 loyalty=71.1 complaints=2 status=Active
  C0070 loyalty=77.1 complaints=1 status=Active
  C0122 loyalty=74.2 complaints=1 status=Active


### READ 2 - InRepair vehicles that were still dispatched (with projection)

This is the dispatch-gate query that should be running before every assignment. Using projection to only return the fields we actually need.

In [ ]:
inrepair_dispatched = db.asset_lifecycle.find(
    {'maintenance_status': 'InRepair', 'performance.total_deliveries': {'$gt': 0}},
    {'_id': 0, 'vehicle_id': 1, 'vehicle_type': 1, 'performance': 1, 'risk_flags.unresolved_incidents': 1}
).sort('performance.fail_pct', -1).limit(10)

print("InRepair vehicles still being dispatched:")
for v in inrepair_dispatched:
    print(f"  {v['vehicle_id']} {v['vehicle_type']:<8} "
          f"deliveries={v['performance']['total_deliveries']:>3} "
          f"fail_pct={v['performance']['fail_pct']:>5}% "
          f"unresolved={v['risk_flags']['unresolved_incidents']}")

InRepair vehicles still being dispatched:
  V011 Diesel   deliveries=  3 fail_pct=100.0% unresolved=0
  V037 CargoVan deliveries=  7 fail_pct= 71.4% unresolved=1
  V075 Diesel   deliveries=  4 fail_pct= 50.0% unresolved=1
  V096 CargoVan deliveries=  6 fail_pct= 50.0% unresolved=2
  V070 CargoVan deliveries=  6 fail_pct= 50.0% unresolved=0
  V002 EV       deliveries=  8 fail_pct= 50.0% unresolved=0
  V057 EV       deliveries=  8 fail_pct= 50.0% unresolved=0
  V109 Hybrid   deliveries= 11 fail_pct= 45.5% unresolved=1
  V017 Hybrid   deliveries= 12 fail_pct= 41.7% unresolved=0
  V034 EV       deliveries=  5 fail_pct= 40.0% unresolved=0


### UPDATE 1 - flag a single customer for retention

In [ ]:
result = db.service_cases.update_one(
    {'customer_id': 'C_TEST_01'},
    {'$set': {
        'retention_flag':   True,
        'priority_action': 'Contact within 48 hours',
        'flagged_at':       datetime.now()
    }}
)
print('matched:', result.matched_count, '| modified:', result.modified_count)

# check it
updated = db.service_cases.find_one({'customer_id': 'C_TEST_01'},
                                     {'customer_id': 1, 'retention_flag': 1, 'priority_action': 1, '_id': 0})
print('after:', updated)

matched: 1 | modified: 1
after: {'customer_id': 'C_TEST_01', 'priority_action': 'Contact within 48 hours', 'retention_flag': True}


### UPDATE 2 - tag all InRepair vehicles with a do-not-dispatch flag

`update_many` lets us apply the same change to many documents at once. This is the operational fix to the InRepair problem - in a real system, dispatch would check this flag before assigning a vehicle.

In [ ]:
result = db.asset_lifecycle.update_many(
    {'maintenance_status': 'InRepair'},
    {'$set': {
        'do_not_dispatch':    True,
        'flag_reason':        'Vehicle in repair - block from assignment',
        'flagged_at':         datetime.now()
    }}
)
print('matched:', result.matched_count, '| modified:', result.modified_count)

matched: 36 | modified: 36


### DELETE - remove the test customer

In [ ]:
result = db.service_cases.delete_one({'customer_id': 'C_TEST_01'})
print('deleted:', result.deleted_count)
print('total docs now:', db.service_cases.count_documents({}))

deleted: 1
total docs now: 650


## 6. Aggregation pipelines

Four pipelines that answer business questions using the embedded structure. Each one would have required multi-table joins in SQL.

### Aggregation 1 - failure rate by pickup zone

`$unwind` flattens the orders array so we can group across orders rather than customers.

In [ ]:
pipeline = [
    {'$unwind': '$orders'},
    {'$match':  {'orders.delivery': {'$ne': None}}},
    {'$group': {
        '_id': '$orders.pickup_zone',
        'deliveries': {'$sum': 1},
        'failed':     {'$sum': {'$cond': [{'$eq': ['$orders.delivery.status', 'Failed']}, 1, 0]}}
    }},
    {'$addFields': {
        'fail_pct': {'$round': [{'$multiply': [{'$divide': ['$failed', '$deliveries']}, 100]}, 1]}
    }},
    {'$sort': {'fail_pct': -1}}
]

print(f"{'zone':<12} {'deliveries':>10} {'failed':>8} {'fail_pct':>10}")
for r in db.service_cases.aggregate(pipeline):
    print(f"  {r['_id']:<10} {r['deliveries']:>10} {r['failed']:>8} {r['fail_pct']:>9}%")

zone         deliveries   failed   fail_pct
  Central           174       33      19.0%
  North             135       22      16.3%
  Riverside         119       18      15.1%
  West              114       14      12.3%
  East              156       19      12.2%
  Airport           113       12      10.6%
  South             139       14      10.1%


Central zone fails at almost double the rate of South - same finding the SQL queries surfaced earlier, but with one query against one collection.

### Aggregation 2 - vehicles with the most unresolved incidents

`$filter` lets us count array elements that match a condition without unwinding.

In [ ]:
pipeline = [
    {'$match': {'incidents': {'$ne': []}}},
    {'$addFields': {
        'unresolved_count': {
            '$size': {
                '$filter': {
                    'input': '$incidents',
                    'as':    'i',
                    'cond':  {'$in': ['$$i.resolution_status', ['Open', 'Escalated']]}
                }
            }
        },
        'total_incidents': {'$size': '$incidents'}
    }},
    {'$match': {'unresolved_count': {'$gt': 0}}},
    {'$sort':  {'unresolved_count': -1}},
    {'$limit': 10},
    {'$project': {'_id': 0, 'vehicle_id': 1, 'vehicle_type': 1, 'maintenance_status': 1,
                  'unresolved_count': 1, 'total_incidents': 1, 'performance.fail_pct': 1}}
]

for r in db.asset_lifecycle.aggregate(pipeline):
    print(f"  {r['vehicle_id']} {r['vehicle_type']:<8} {r['maintenance_status']:<10} "
          f"unresolved={r['unresolved_count']}/{r['total_incidents']} "
          f"fail_pct={r['performance']['fail_pct']}%")

  V108 Diesel   InRepair   unresolved=4/7 fail_pct=33.3%
  V030 CargoVan Active     unresolved=3/6 fail_pct=13.3%
  V111 EV       Active     unresolved=3/3 fail_pct=0.0%
  V112 EV       InRepair   unresolved=3/4 fail_pct=28.6%
  V009 CargoVan Active     unresolved=3/5 fail_pct=0.0%
  V088 Diesel   InRepair   unresolved=3/5 fail_pct=15.4%
  V046 EV       Active     unresolved=3/6 fail_pct=10.0%
  V042 EV       InRepair   unresolved=2/5 fail_pct=23.1%
  V005 CargoVan Active     unresolved=2/5 fail_pct=15.4%
  V047 EV       Scheduled  unresolved=2/9 fail_pct=0.0%


### Aggregation 3 - data integrity issue rate by hub

Using the integrity flags we embedded inside each delivery.

In [ ]:
pipeline = [
    {'$unwind': '$orders'},
    {'$match':  {'orders.delivery': {'$ne': None}}},
    {'$group': {
        '_id': '$orders.delivery.hub_id',
        'deliveries':        {'$sum': 1},
        'integrity_issues':  {'$sum': {'$cond': ['$orders.delivery.integrity_flags.any_issue', 1, 0]}},
        'negative_duration': {'$sum': {'$cond': ['$orders.delivery.integrity_flags.negative_duration', 1, 0]}},
        'failed_but_rated':  {'$sum': {'$cond': ['$orders.delivery.integrity_flags.failed_but_rated', 1, 0]}}
    }},
    {'$addFields': {
        'integrity_pct': {'$round': [{'$multiply': [{'$divide': ['$integrity_issues', '$deliveries']}, 100]}, 1]}
    }},
    {'$sort': {'integrity_pct': -1}}
]

print(f"{'hub':<6} {'deliveries':>10} {'issues':>8} {'integrity_pct':>14}")
for r in db.service_cases.aggregate(pipeline):
    print(f"  {r['_id']:<4} {r['deliveries']:>10} {r['integrity_issues']:>8} {r['integrity_pct']:>13}%")

hub    deliveries   issues  integrity_pct
  H08         128       35          27.3%
  H05         115       28          24.3%
  H01         136       33          24.3%
  H06         104       24          23.1%
  H04         127       26          20.5%
  H03         119       23          19.3%
  H07         115       20          17.4%
  H02         106       18          17.0%


H08 (Midtown Relay) has the worst integrity issue rate at over 27%. Same hub that had the highest failure rate in the SQL analysis.

### Aggregation 4 - app session quality by device

How does success rate and latency vary by device type? Useful for the customer experience team.

In [ ]:
pipeline = [
    {'$group': {
        '_id':         '$device_type',
        'sessions':    {'$sum': 1},
        'avg_events':  {'$avg': '$event_count'},
        'avg_success': {'$avg': '$success_rate'},
        'avg_latency': {'$avg': '$avg_latency'}
    }},
    {'$project': {
        'sessions':    1,
        'avg_events':  {'$round': ['$avg_events', 2]},
        'avg_success': {'$round': ['$avg_success', 1]},
        'avg_latency': {'$round': ['$avg_latency', 0]}
    }},
    {'$sort': {'sessions': -1}}
]

print(f"{'device':<10} {'sessions':>10} {'avg_events':>12} {'success%':>10} {'latency_ms':>12}")
for r in db.platform_events.aggregate(pipeline):
    print(f"  {r['_id']:<8} {r['sessions']:>10} {r['avg_events']:>12} {r['avg_success']:>9}% {r['avg_latency']:>11}")

device       sessions   avg_events   success%   latency_ms
  Android         315          1.0      93.3%       464.0
  iOS             231          1.0      95.2%       463.0
  Web              91         1.02      93.4%       469.0


## 7. Indexes

Indexes tell MongoDB which fields to build lookup structures on so it doesn't scan every document. Without them, every query does a `COLLSCAN` (full collection scan) - fine for our 650 customers, but catastrophic at production scale.

I'm creating indexes based on the actual queries used in this notebook and in the SQL analysis - so each index has a real workload it supports, not just being defensive.

In [ ]:
# service_cases
db.service_cases.create_index('customer_id', unique=True)         # primary key, unique constraint
db.service_cases.create_index('loyalty_score')                    # range queries for retention analysis
db.service_cases.create_index('account_status')                   # filter dormant/suspended customers

# asset_lifecycle
db.asset_lifecycle.create_index('vehicle_id', unique=True)
db.asset_lifecycle.create_index('maintenance_status')             # the dispatch-gate query
# compound index for the "low battery + high mileage" risk query
db.asset_lifecycle.create_index([('battery_health_pct', 1), ('odometer_km', 1)])

# platform_events
db.platform_events.create_index('session_id', unique=True)
db.platform_events.create_index('customer_id')                    # lookup all sessions for a customer
db.platform_events.create_index('started_at')                     # time-range queries

print("indexes created on all 3 collections")
print("\n--- listing indexes on each collection ---")
for coll_name in ['service_cases', 'asset_lifecycle', 'platform_events']:
    print(f"\n{coll_name}:")
    for idx in db[coll_name].list_indexes():
        print(f"  {idx['name']:<35} keys: {dict(idx['key'])}")

indexes created on all 3 collections

--- listing indexes on each collection ---

service_cases:
  _id_                                keys: {'_id': 1}
  customer_id_1                       keys: {'customer_id': 1}
  loyalty_score_1                     keys: {'loyalty_score': 1}
  account_status_1                    keys: {'account_status': 1}

asset_lifecycle:
  _id_                                keys: {'_id': 1}
  vehicle_id_1                        keys: {'vehicle_id': 1}
  maintenance_status_1                keys: {'maintenance_status': 1}
  battery_health_pct_1_odometer_km_1  keys: {'battery_health_pct': 1, 'odometer_km': 1}

platform_events:
  _id_                                keys: {'_id': 1}
  session_id_1                        keys: {'session_id': 1}
  customer_id_1                       keys: {'customer_id': 1}
  started_at_1                        keys: {'started_at': 1}


### Why these indexes

**`maintenance_status` on `asset_lifecycle`** is the most operationally important one. The dispatch-gate query - find all InRepair vehicles before assigning a job - runs on this field. Without the index, MongoDB scans all 120 vehicle documents every time. With it, it goes directly to the matching ones via an IXSCAN.

**`loyalty_score` on `service_cases`** supports range queries like "loyalty > 70". Without an index, a range query has to check every document - even ones that obviously don't match. The index lets MongoDB jump straight to the lower bound and stop at the upper bound.

**Compound `battery_health_pct + odometer_km` on `asset_lifecycle`** supports the "low battery and high mileage" risk query. A single compound index covers both conditions in one scan, which is more efficient than two separate single-field indexes that MongoDB would have to intersect.

**Unique indexes on the id fields** enforce the data integrity rule that each entity has exactly one document.

In production at much larger scale, I'd add a partial index on `risk_flags.is_in_repair=true` since the InRepair set is a small fraction of the fleet and a partial index would be a fraction of the size of the full one.

## 8. Query optimisation - explain plans

Using `.explain("executionStats")` to compare query performance with and without indexes. The key fields to look at:

- **winning stage**: `COLLSCAN` (no index used, full scan) vs `IXSCAN` (index scan, jumps directly to matches)
- **totalDocsExamined**: how many documents MongoDB had to read
- **nReturned**: how many actually matched

If `totalDocsExamined` is much bigger than `nReturned`, the query is doing wasted work.

### Test 1 - `maintenance_status` index on `asset_lifecycle`

In [ ]:
# drop the index temporarily so we can see what happens without it
try:
    db.asset_lifecycle.drop_index('maintenance_status_1')
    print("dropped index for testing")
except OperationFailure:
    print("index already absent")

# BEFORE - no index
explain_before = db.command(
    'explain',
    {'find': 'asset_lifecycle', 'filter': {'maintenance_status': 'InRepair'}},
    verbosity='executionStats'
)
stats_before = explain_before['executionStats']
stage_before    = stats_before['executionStages']['stage']
scanned_before  = stats_before['totalDocsExamined']
returned_before = stats_before['nReturned']

print("WITHOUT index:")
print(f"  stage:     {stage_before}")
print(f"  scanned:   {scanned_before}")
print(f"  returned:  {returned_before}")
print(f"  wasted:    {scanned_before - returned_before} documents read but discarded")

dropped index for testing
WITHOUT index:
  stage:     COLLSCAN
  scanned:   120
  returned:  36
  wasted:    84 documents read but discarded


In [ ]:
# put the index back
db.asset_lifecycle.create_index('maintenance_status')

# AFTER - with index
explain_after = db.command(
    'explain',
    {'find': 'asset_lifecycle', 'filter': {'maintenance_status': 'InRepair'}},
    verbosity='executionStats'
)
stats_after = explain_after['executionStats']
stage_after    = stats_after['executionStages']['stage']
scanned_after  = stats_after['totalDocsExamined']
returned_after = stats_after['nReturned']

print("WITH index:")
print(f"  stage:     {stage_after}")
print(f"  scanned:   {scanned_after}")
print(f"  returned:  {returned_after}")
print()
print("comparison:")
print(f"  before: {stage_before:<12} scanned={scanned_before:>4} returned={returned_before}")
print(f"  after:  {stage_after:<12} scanned={scanned_after:>4} returned={returned_after}")

WITH index:
  stage:     FETCH
  scanned:   36
  returned:  36

comparison:
  before: COLLSCAN     scanned= 120 returned=36
  after:  FETCH        scanned=  36 returned=36


The change from `COLLSCAN` to `IXSCAN` confirms the index is being used. At this collection size (120 vehicles) the absolute scan count is small, but the ratio matters: without the index, MongoDB reads every document; with it, it reads only matching documents. At 10,000 or 1 million vehicles this is the difference between an instant query and a timeout.

### Test 2 - `loyalty_score` range query on `service_cases`

Range queries on unindexed fields are particularly expensive because MongoDB can't know where matching documents are without checking them all.

In [ ]:
try:
    db.service_cases.drop_index('loyalty_score_1')
    print("dropped index for testing")
except OperationFailure:
    print("index already absent")

explain_before = db.command(
    'explain',
    {'find': 'service_cases', 'filter': {'loyalty_score': {'$gt': 70}}},
    verbosity='executionStats'
)
stats = explain_before['executionStats']
stage_before    = stats['executionStages']['stage']
scanned_before  = stats['totalDocsExamined']
returned_before = stats['nReturned']

print("WITHOUT index (range query loyalty_score > 70):")
print(f"  stage:    {stage_before}")
print(f"  scanned:  {scanned_before}")
print(f"  returned: {returned_before}")

dropped index for testing
WITHOUT index (range query loyalty_score > 70):
  stage:    COLLSCAN
  scanned:  650
  returned: 162


In [ ]:
db.service_cases.create_index('loyalty_score')

explain_after = db.command(
    'explain',
    {'find': 'service_cases', 'filter': {'loyalty_score': {'$gt': 70}}},
    verbosity='executionStats'
)
stats = explain_after['executionStats']
stage_after    = stats['executionStages']['stage']
scanned_after  = stats['totalDocsExamined']
returned_after = stats['nReturned']

print("WITH index:")
print(f"  stage:    {stage_after}")
print(f"  scanned:  {scanned_after}")
print(f"  returned: {returned_after}")
print()
reduction = (scanned_before - scanned_after) / scanned_before * 100 if scanned_before > 0 else 0
print(f"scan reduction: {scanned_before} -> {scanned_after} ({reduction:.0f}% fewer docs read)")

WITH index:
  stage:    FETCH
  scanned:  162
  returned: 162

scan reduction: 650 -> 162 (75% fewer docs read)


The index reduces the scan from a full 650 customers to roughly the number of matching customers. The improvement scales with how selective the filter is - the more documents you're excluding, the more an index pays off.

### Test 3 - compound index on `asset_lifecycle`

For queries that filter on two fields together, a compound index over both can be much faster than two separate single-field indexes. Below we test the compound query (low battery AND high mileage) first **without** the compound index, then with it.

In [ ]:
# query: low-battery, high-mileage vehicles (the predictive risk query)
query = {'battery_health_pct': {'$lt': 70}, 'odometer_km': {'$gt': 100000}}

# BEFORE - drop the compound index so we can see the COLLSCAN plan
try:
    db.asset_lifecycle.drop_index('battery_health_pct_1_odometer_km_1')
    print("dropped compound index for testing")
except OperationFailure:
    print("compound index already absent")

explain_before = db.command('explain',
    {'find': 'asset_lifecycle', 'filter': query},
    verbosity='executionStats')
stats = explain_before['executionStats']
stage_before    = stats['executionStages']['stage']
scanned_before  = stats['totalDocsExamined']
returned_before = stats['nReturned']

print("WITHOUT compound index:")
print(f"  stage:    {stage_before}")
print(f"  scanned:  {scanned_before}")
print(f"  returned: {returned_before}")

dropped compound index for testing
WITHOUT compound index:
  stage:    COLLSCAN
  scanned:  120
  returned: 19


In [ ]:
# AFTER - recreate the compound index and run the query again
db.asset_lifecycle.create_index([('battery_health_pct', 1), ('odometer_km', 1)])

explain_after = db.command('explain',
    {'find': 'asset_lifecycle', 'filter': query},
    verbosity='executionStats')
stats = explain_after['executionStats']
stage_after    = stats['executionStages']['stage']
scanned_after  = stats['totalDocsExamined']
returned_after = stats['nReturned']

print("WITH compound index:")
print(f"  stage:    {stage_after}")
print(f"  scanned:  {scanned_after}")
print(f"  returned: {returned_after}")
print()
print("comparison:")
print(f"  before: {stage_before:<12} scanned={scanned_before:>4} returned={returned_before}")
print(f"  after:  {stage_after:<12} scanned={scanned_after:>4} returned={returned_after}")

WITH compound index:
  stage:    FETCH
  scanned:  19
  returned: 19

comparison:
  before: COLLSCAN     scanned= 120 returned=19
  after:  FETCH        scanned=  19 returned=19


The compound index switches the query from `COLLSCAN` to `IXSCAN` for both filter conditions in one pass. If we had used two separate single-field indexes instead, MongoDB would either pick one and filter the other in memory, or attempt an index intersection - both slower than a single compound index that orders matching documents on both keys together.

## 9. Summary

The MongoDB design takes the relationships that were spread across multiple SQL tables and packages them into documents that match how the business actually thinks about its data.

The three collections are designed around real business views:

- `service_cases` answers customer experience questions in one document read - orders, deliveries, complaints, incidents all nested together. This is what management asked for when they said they wanted an "integrated operational view".
- `asset_lifecycle` makes the InRepair-dispatch problem visible in a single query that was previously a 3-table join across vehicles, deliveries, and incidents.
- `platform_events` reshapes the flat app event log into the session-oriented structure it naturally is.

The CRUD demonstrations show how these documents support real operational queries - finding at-risk customers, blocking InRepair vehicle dispatch, flagging individual customers for retention action. None of these are toy operations.

The aggregation pipelines produce the same findings as the SQL analysis (Central zone has the worst failure rate, H08 has the worst integrity issues) but in fewer steps and from a single collection.

The indexes are picked based on the actual queries used, not added defensively. The explain plans confirm they change the query stage from COLLSCAN to IXSCAN, which is what we want.

This completes the three technical sections of the assignment - Python, R, MongoDB - and the findings line up across all three.